# RNA Structure Processing Tutorial

This notebook demonstrates how to use the RNA structure processing pipeline for various tasks.

## Setup

First, let's import the necessary modules and set up our environment.

In [2]:
import sys
sys.path.append('.')

from utils.pdb_downloader import PDBDownloader
# from utils.pdb_to_npy import PDBToNumpyConverter
# from utils.npy_to_pdb import NumpyToPDBConverter
from utils.extract_loops import LoopExtractor
from utils.extract_rna_segments import RNAExtractor
from utils.extract_sequence import SequenceExtractor
import numpy as np
# import matplotlib.pyplot as plt
from pathlib import Path
import os

## A1. Data Acquisition

### Downloading RNA structures from PDB

In [6]:
# Initialize the downloader
downloader = PDBDownloader(output_dir="/Users/ju/Documents/Dev/pdb_prune/experiment/exp5/CASP16_data")

# Example 1: Download specific PDB IDs
pdb_ids = ['8uo6','9cfn','9c2k','9c2k','9dcf','9dcf','9b0l','9b0l','9ely','9dxd','9dxd','9cbu','9cbx','9bzc','9bz1','8vqv','8vvj','9j6y','9mee','9c75','9isv','9j3r','9j3t','9c75','9mcw','9bzc','9bz1','8vqv','8vvj' ]# downloader.download_pdbs(pdb_ids)
downloader.download_pdbs(pdb_ids)
# # Example 2: Search and download by criteria
# criteria = {
#     'resolution': 3.0,  # Maximum resolution
#     'rna_only': True,  # RNA-only structures
#     'min_length': 50   # Minimum sequence length
# }
# downloader.search_and_download(criteria)

# Example 3: Download from file
#downloader.download_from_file("pdb_list.txt", max_pdbs=10)



File already exists: /Users/ju/Documents/Dev/pdb_prune/experiment/exp5/CASP16_data/8uo6.pdb
File already exists: /Users/ju/Documents/Dev/pdb_prune/experiment/exp5/CASP16_data/9cfn.pdb
File already exists: /Users/ju/Documents/Dev/pdb_prune/experiment/exp5/CASP16_data/9c2k.pdb
File already exists: /Users/ju/Documents/Dev/pdb_prune/experiment/exp5/CASP16_data/9c2k.pdb
File already exists: /Users/ju/Documents/Dev/pdb_prune/experiment/exp5/CASP16_data/9dcf.pdb
File already exists: /Users/ju/Documents/Dev/pdb_prune/experiment/exp5/CASP16_data/9dcf.pdb
File already exists: /Users/ju/Documents/Dev/pdb_prune/experiment/exp5/CASP16_data/9b0l.pdb
File already exists: /Users/ju/Documents/Dev/pdb_prune/experiment/exp5/CASP16_data/9b0l.pdb
File already exists: /Users/ju/Documents/Dev/pdb_prune/experiment/exp5/CASP16_data/9ely.pdb
Error downloading 9dxd: 404 Client Error: Not Found for url: https://files.rcsb.org/download/9dxd.cif
Error downloading 9dxd: 404 Client Error: Not Found for url: https://

In [ ]:
   # Download specific PDB IDs with limit
   !python pdb_downloader.py --pdb-ids 1ABC 2XYZ --max-pdbs 5

   # Download from file with limit
   !python pdb_downloader.py --input-file pdb_list.txt --max-pdbs 10

   # Search and download with limit
   !python pdb_downloader.py --search --rna-only --max-pdbs 20

   # Resume download from last successful file
   !python utils/resume_pdb_search.py --output_dir data/raw_data/pdbs/raw_pdbs_full_download --max_entries 10

In [7]:
from utils.pdb_downloader import PDBDownloader

LIST_FILE = "/Users/xiaojuzhang/Dev/pdb_prune/experiment/exp3/validation_pdbs_ids.txt"
OUTPUT_DIR = "/Users/xiaojuzhang/Dev/pdb_prune/experiment/exp3/validation_downloads"

downloader = PDBDownloader(output_dir=OUTPUT_DIR)
downloader.download_from_file(LIST_FILE) 



File already exists: /Users/xiaojuzhang/Dev/pdb_prune/experiment/exp3/validation_downloads/7v6c.pdb
File already exists: /Users/xiaojuzhang/Dev/pdb_prune/experiment/exp3/validation_downloads/6ifk.pdb
Downloaded: /Users/xiaojuzhang/Dev/pdb_prune/experiment/exp3/validation_downloads/7uqb.cif
File already exists: /Users/xiaojuzhang/Dev/pdb_prune/experiment/exp3/validation_downloads/4c4w.pdb
File already exists: /Users/xiaojuzhang/Dev/pdb_prune/experiment/exp3/validation_downloads/3ssf.pdb
File already exists: /Users/xiaojuzhang/Dev/pdb_prune/experiment/exp3/validation_downloads/7eni.pdb
File already exists: /Users/xiaojuzhang/Dev/pdb_prune/experiment/exp3/validation_downloads/3jcm.pdb
File already exists: /Users/xiaojuzhang/Dev/pdb_prune/experiment/exp3/validation_downloads/7dlz.pdb
Downloaded: /Users/xiaojuzhang/Dev/pdb_prune/experiment/exp3/validation_downloads/7pwg.cif
Downloaded: /Users/xiaojuzhang/Dev/pdb_prune/experiment/exp3/validation_downloads/7ohq.cif
File already exists: /Use

In [8]:
from pathlib import Path

list_file   = Path("/Users/xiaojuzhang/Dev/pdb_prune/experiment/exp3/validation_pdbs_ids.txt")
download_dir = Path("/Users/xiaojuzhang/Dev/pdb_prune/experiment/exp3/validation_downloads")

# Read expected IDs (strip whitespace and skip empty lines)
expected = {line.strip().upper() for line in list_file.read_text().splitlines() if line.strip()}

# Collect already-downloaded prefixes (handles both .pdb and .cif extensions)
downloaded = {p.stem.upper() for p in download_dir.glob("*") if p.suffix.lower() in {".pdb", ".cif"}}

missing = sorted(expected - downloaded)
print(f"Missing ({len(missing)}) IDs:")
for pdb_id in missing:
    print(pdb_id)

Missing (0) IDs:


- 15 IDs appear more than once in the list (duplicates), so the raw line count is 221.
- After removing duplicates there are only 206 unique PDB IDs expected.
- You already have 206 files in validation_downloads , so nothing is missing —you simply downloaded every unique entry.

## A2. PDB RNA Chains Extraction

In [12]:
!python utils/pdb_rna_processor.py --original-dir ~/Documents/Dev/pdb_prune/experiment/exp5/CASP16_data \
                                   --processed-dir ~/Documents/Dev/pdb_prune/experiment/exp5/CASP16_data/validation_extracted_pdbs \
                                   --fasta-dir ~/Documents/Dev/pdb_prune/experiment/exp5/CASP16_data/validation_extracted_sequences


2025-12-22 12:03:17,155 - INFO - Found 19 structure files to process (atom_option=backbone)
2025-12-22 12:03:17,155 - INFO - 
Processing structure: 9dcf.pdb (atom_option=backbone)
2025-12-22 12:03:17,155 - INFO - --------------------------------------------------
2025-12-22 12:03:17,222 - INFO - Found 1 RNA chain(s):
2025-12-22 12:03:17,223 - INFO - - /Users/ju/Documents/Dev/pdb_prune/experiment/exp5/CASP16_data/validation_extracted_pdbs/9dcf_C.pdb
2025-12-22 12:03:17,227 - INFO -   File: /Users/ju/Documents/Dev/pdb_prune/experiment/exp5/CASP16_data/validation_extracted_pdbs/9dcf_C.pdb | Residues: 77 | Atoms: 539 | Chain: C
2025-12-22 12:03:17,227 - INFO - 
Processing structure: 9bzc.pdb (atom_option=backbone)
2025-12-22 12:03:17,227 - INFO - --------------------------------------------------
2025-12-22 12:03:17,269 - INFO - Found 1 RNA chain(s):
2025-12-22 12:03:17,269 - INFO - - /Users/ju/Documents/Dev/pdb_prune/experiment/exp5/CASP16_data/validation_extracted_pdbs/9bzc_A.pdb
2025-12

## B. PDB download from RFAM

In [21]:
import requests
rf_id = "RF00001"
response = requests.get(f"https://rfam.org/family/{rf_id}/structures?content-type=application/json")
data = response.json()
for entry in data:
    print(f"PDB: {entry['pdb_id']}, Chain: {entry['chain']}, Resolution: {entry['resolution']}")

TypeError: string indices must be integers